# exp-back — ex2: compose log_back ∘ exp_back — recover identity on exp(log(x))

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `exp-back`. Running the final beacon cell reports progress against the `Backprop: exp_back` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: exp_back` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`exp-back`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "exp-back"
DD_SUBTOPIC = "Backprop: exp_back"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `exp_back` composed with `log_back` — quick refresher

ex1 derived `exp_back = grad_out * out`. The deeper facet is what happens when you COMPOSE inverse ops in a chain.

For `y = exp(log(x))` the math is `y = x` so `dy/dx = 1`. Through the chain rule applied via the back fns:

```
u = log(x)         out_u = log(x)
y = exp(u)         out_y = exp(u) = x

exp_back(g, out_y, u)  =  g * out_y  =  g * x
log_back(g', out_u, x) =  g' / x

Compose:  g_x = log_back(exp_back(g, x, log(x)), log(x), x)
              = (g * x) / x  =  g
```

The product `out_exp / x_leaf` collapses to **1** — the back-fn pipeline produces the identity gradient, mirroring the analytic derivative.

### Exercise 2 — compose log_back ∘ exp_back — recover identity on exp(log(x))

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply two-step chain composition: compose log_back and exp_back to verify that the gradient through exp(log(x)) equals the seed grad_out.
> Keywords: compose, chain-rule, inverse-ops, identity
> ```

**KCs targeted:** `chain-rule-elementwise`, `back-fn-uses-cached-out`

Implement TWO back fns plus the composed chain:

1. `exp_back(grad_out, out, x)` — `grad_out * out` (cached out reuse).
2. `log_back(grad_out, out, x)` — `grad_out / x`.
3. `chain_exp_of_log(grad_out, x_leaf)` — walk the reverse pass of `y = exp(log(x_leaf))` using your two back fns. Return the leaf grad.

The pipeline:
```
u    = log(x_leaf)                              # forward
y    = exp(u)                                   # forward
g_u  = exp_back(grad_out, y, u)                 # = grad_out * y
g_x  = log_back(g_u, u, x_leaf)                 # = g_u / x_leaf
       = grad_out * y / x_leaf
       = grad_out                               # because y = x_leaf
```

The point: the final leaf grad must equal `grad_out` exactly — the two-step chain through inverse ops collapses to the identity. This is the cleanest demonstration that back fns COMPOSE by ordinary function composition (no extra accumulation in a single-parent chain).

Tests verify:
- both individual back fns,
- the composed chain returns `grad_out` for positive `x_leaf` of varying values and grad_out of varying values,
- agreement with torch.autograd on `exp(log(x)).sum()`.

No autograd inside your implementation.

In [ ]:
def exp_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    return grad_out * out


def log_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    return grad_out / x


def chain_exp_of_log(grad_out: Tensor, x_leaf: Tensor) -> Tensor:
    # forward (cached for back-fn use)
    u = t.log(x_leaf)
    y = t.exp(u)
    # reverse: exp_back, then log_back
    g_u = exp_back(grad_out, y, u)
    g_x = log_back(g_u, u, x_leaf)
    return g_x


<details><summary>Solution</summary>

```python
def exp_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    return grad_out * out


def log_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    return grad_out / x


def chain_exp_of_log(grad_out: Tensor, x_leaf: Tensor) -> Tensor:
    # forward (cached for back-fn use)
    u = t.log(x_leaf)
    y = t.exp(u)
    # reverse: exp_back, then log_back
    g_u = exp_back(grad_out, y, u)
    g_x = log_back(g_u, u, x_leaf)
    return g_x
```

**Why the chain collapses to identity.** `exp_back` multiplies by `y = exp(log(x)) = x`, then `log_back` divides by `x`. The multiplications cancel: `grad_out * x / x = grad_out`. This is the back-fn-level mirror of the analytic fact that `d/dx exp(log(x)) = 1`.

**Why cache `y` in the chain.** `exp_back` needs the cached `out` of the exp call. If you wrote `g_u = exp_back(grad_out, t.exp(u), u)` you'd recompute exp redundantly. Saving `y` once on the forward matches what `Recipe.args` would do in a real autograd graph.

**Numerical drift.** The composition isn't bit-exact — `log(x)` followed by `exp(.)` introduces one rounding step. The `atol=1e-3` on the random-scale test accounts for it; at unit scale, `atol=1e-5` is fine.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()